# 03 · The covered-call book · Options Surface Lab

**Assignment 2.** The weekly loop, the blotter and the Reg T ledger, walked through against
the raw bars they came from.

> **Sections 1–4 are T-66 and are not written yet.** They walk one week end to end (the raw
> bars beside the blotter rows), explain the skip log, and show the NAV identity. The
> material exists — `tests/covered_call/test_engine.py` hand-checks 2026-W37 in a docstring,
> and `Book.weekly` is the week-by-week table — it has simply not been lifted in here.
> **Section 5 below is T-58** and stands on its own: it reads the tape, not the book.


In [1]:
import sys
from pathlib import Path

# repo root on sys.path whether the kernel starts in notebooks/ or the repo root
_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import plotly.graph_objects as go

from options_surface_lab.covered_call.evidence import (
    NON_SIMULTANEITY_CAVEAT,
    REFUSAL_ORDER,
    mid_vs_print,
)
from options_surface_lab.covered_call.rules import Params
from options_surface_lab.covered_call.tape import TAPE_PATH, load_tape
from options_surface_lab.covered_call.engine import run_backtest

pd.set_option("display.width", 120)

params = Params()
tape = load_tape(TAPE_PATH, fallback=False)
print(f"tape: {len(tape.bars):,} bars, synthetic={tape.synthetic}")
print(f"window: {params.start} -> {params.end}, band ±{params.ntm_band:.0%}")

tape: 37,857 bars, synthetic=False
window: 2026-07-06 -> 2026-09-11, band ±5%


## 5 · Mid vs print — what the fill assumption rests on (FR-17, SPEC §10)

The backtest fills at the **midpoint of the bar's final bid and ask**. That is an assumption,
and the brief asks for it to be justified rather than asserted. The justification available
from this tape is direct: on the bars where somebody *did* trade, how far was the print from
the mid we would have filled at?

Two things to keep in view while reading, because they bound what the answer can mean:

* **The pair is not simultaneous.** Inside an hourly bar the print is the last trade and the
  quote is the last bid and ask reported. They are the same *bar*, not the same *instant* —
  and there is no timestamp for the quote update to check (SPEC §6.3, struck 2026-09-14).
* **The sample is near-the-money calls of the regular session**, ±`params.ntm_band` of that
  bar's own spot, because that is where the rule writes. A fit over the whole chain would be
  carried by deep-OTM contracts quoted in pennies that the strategy never touches — and the
  16:00 ET post-close bar is excluded for the same reason `rules.CLOSING_BAR_HOUR_ET` exists
  (T-62): it is a stub on a fifth of the volume, priced against an extended-hours spot.

`covered_call.evidence.mid_vs_print` is the transform; everything below is read off its
output.

In [2]:
evidence = mid_vs_print(tape, params)
print(evidence.describe())

n = 16626 of 37073 QQQ.O option bars in the window, near the money (±5% of spot), regular session only: print = 0.9979 × mid + 0.0104, R² = 0.9962; median |print − mid| = $0.035; median of |print − mid| / mid = 1.89%.
Refused: not_a_call 0, no_strike 0, post_close 4633, no_spot 0, outside_band 10178, no_quote 1492, no_print 4144.
Reconciles: 16626 + 20447 = 37073.


### 5.1 · The sample, and what it refused

`n` on its own is a number without a denominator. Every option bar the window holds lands in
exactly one bucket — in the sample, or refused for one named reason — and the tally
reconciles. The order is **structure, then market**: a bar that is not a call, has no strike,
or is a post-close stub is refused before anything is asked about its quote, because none of
those is a fact about the market. Then the band, because it is the sample *universe* rather
than a failure — which leaves `no_quote` and `no_print` describing the near-the-money bars
only, and that is the question worth asking.

In [3]:
rows = [("in the sample (n)", evidence.n)]
rows += [(f"refused: {name}", evidence.refusals[name]) for name in REFUSAL_ORDER]
tally = pd.DataFrame(rows, columns=["bucket", "bars"])
tally["share"] = (tally["bars"] / evidence.considered).map("{:.1%}".format)
near = evidence.n + evidence.refusals["no_quote"] + evidence.refusals["no_print"]

print(tally.to_string(index=False))
print(f"\ntotal option bars in the window: {evidence.considered:,}")
print(f"reconciles: {evidence.n + sum(evidence.refusals.values()):,}")
print(f"\nof the {near:,} near-the-money bars, "
      f"{evidence.n / near:.1%} carried both a valid quote and a print")

               bucket  bars share
    in the sample (n) 16626 44.8%
  refused: not_a_call     0  0.0%
   refused: no_strike     0  0.0%
  refused: post_close  4633 12.5%
     refused: no_spot     0  0.0%
refused: outside_band 10178 27.5%
    refused: no_quote  1492  4.0%
    refused: no_print  4144 11.2%

total option bars in the window: 37,073
reconciles: 37,073

of the 22,262 near-the-money bars, 74.7% carried both a valid quote and a print


### 5.2 · The fit

OLS of the print on the mid, with `y = x` drawn through it. A slope of one and an intercept of
zero would say the mid is an unbiased estimate of where the contract actually traded in that
hour; the scatter says how tightly.

In [4]:
pts = evidence.points
lo, hi = float(pts["mid"].min()), float(pts["mid"].max())

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=pts["mid"], y=pts["trdprc_1"], mode="markers", name="contract-bars",
    marker=dict(size=3, opacity=0.35),
    hovertemplate="mid %{x:$.2f}<br>print %{y:$.2f}<extra></extra>",
))
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="y = x",
                         line=dict(dash="dash", width=1)))
fig.add_trace(go.Scatter(x=[lo, hi], y=evidence.fit_y([lo, hi]), mode="lines",
                         name=f"OLS (R² = {evidence.r2:.4f})", line=dict(width=2)))
fig.update_layout(
    title="Print vs mid — near-the-money QQQ calls, hourly bars",
    xaxis_title="mid = (bid + ask) / 2 at the bar",
    yaxis_title="trdprc_1 — the last print in the bar",
    height=520, template="plotly_white",
)
fig.show()

print(evidence.headline())

n = 16626 of 37073 QQQ.O option bars in the window, near the money (±5% of spot), regular session only: print = 0.9979 × mid + 0.0104, R² = 0.9962; median |print − mid| = $0.035; median of |print − mid| / mid = 1.89%.


The cloud is tight but it is not a line, and the residual is not uniform: the dollar gap
grows with the contract's price, which is what you would expect when the spread does too.
That is the reason the median gap is quoted **both** in dollars and as a percentage of the
mid — one number is what the book pays, the other is what the market's own width costs.

**Read the R² with that in mind.** ±5% of a $720 underlying is ±$36, so the sample reaches
contracts more than thirty dollars in the money whose mid is nearly all intrinsic value and
whose price is therefore almost mechanically predictable. Two measurements make that concrete,
and both belong in the write-up:

* Bars with mid > $10 are about a third of the sample and carry **two thirds of the fit's
  leverage**; the \$5–\$9.50 band where all ten fills actually sat carries half a percent of it.
* The R² of the fitted line and the R² of `y = x` **with no fitting at all** agree to the sixth
  decimal. The OLS is not discovering anything the identity line did not already say.

So the R² is worth reporting because the brief asks for it, but it is not the evidence. What
bounds the backtest is the size of the gap relative to the market's own width — §5.2a — and
the priced worst case in §5.5.

### 5.2a · The scale that makes a gap meaningful is the spread

Three and a half cents sounds small. Measured against the quoted bid-ask of the very same bar,
it is not small at all: the median `|print − mid|` divided by that bar's own **half-spread** is
almost exactly **1.0**. The typical print sits a full half-spread from the mid — which is to
say, at a quote edge rather than near the middle.

The distribution below says the same thing without the ratio. Fewer than a third of prints land
strictly inside the quoted market; more than a third land outside it altogether, which is what
you would expect when the last trade of an hour is compared with the last quote of that hour
and the two are minutes apart.

**This does not invalidate the fill assumption — it re-scales the claim.** The mid is
*unbiased* (§5.5), not *accurate*. Filling a seller at the mid is optimistic by roughly half a
spread on any single trade, and the right question is what that is worth over ten trades, which
§5.5 prices outright.

In [5]:
half = (pts["ask"] - pts["bid"]) / 2.0
inside = ((pts["trdprc_1"] > pts["bid"]) & (pts["trdprc_1"] < pts["ask"])).mean()
at_edge = ((pts["trdprc_1"] == pts["bid"]) | (pts["trdprc_1"] == pts["ask"])).mean()
outside = ((pts["trdprc_1"] < pts["bid"]) | (pts["trdprc_1"] > pts["ask"])).mean()

print(f"median |print - mid| / half-spread, per row: {(pts['abs_gap'] / half).median():.3f}")
print(f"median half-spread: ${half.median():.4f}   median |gap|: ${pts['abs_gap'].median():.4f}")
print()
print(f"print strictly inside the quoted market: {inside:.1%}")
print(f"print exactly at the bid or the ask:     {at_edge:.1%}")
print(f"print outside the quoted market:         {outside:.1%}")

median |print - mid| / half-spread, per row: 1.000
median half-spread: $0.0250   median |gap|: $0.0350

print strictly inside the quoted market: 30.4%
print exactly at the bid or the ask:     28.3%
print outside the quoted market:         41.3%


In [6]:
by_price = pts.assign(bucket=pd.cut(
    pts["mid"], [0, 1, 2, 5, 10, 1e9],
    labels=["< $1", "$1–2", "$2–5", "$5–10", "> $10"],
))
summary = by_price.groupby("bucket", observed=True).agg(
    bars=("mid", "size"),
    median_mid=("mid", "median"),
    median_abs_gap=("abs_gap", "median"),
    median_gap_pct=("gap_pct", "median"),
).round(3)
print(summary.to_string())

        bars  median_mid  median_abs_gap  median_gap_pct
bucket                                                  
< $1    5057       0.135           0.005           5.882
$1–2    1188       1.448           0.020           1.261
$2–5    2230       3.315           0.025           0.744
$5–10   2479       7.295           0.075           1.018
> $10   5672      18.188           0.340           1.866


### 5.3 · Does the answer depend on the band?

`ntm_band` is a parameter the page prints (FR-14), so it is worth knowing whether the
conclusion is an artifact of where it was set. Sweeping it: the sample triples, and the slope,
the R² and the median gap barely move. The fill assumption is not being held up by the choice
of band.

Two details the first version of this notebook got wrong, both caught by review. The R² does
**not** move monotonically — it dips to its floor at ±2%, where the sample is tightest around
the money, so quoting the ±1% row as the floor understates the range. And above **±14%** the
band stops binding at all: every wider band returns the identical sample, so a sweep quoted
"to ±25%" is the same measurement repeated. The grid below runs to the saturation point and
prints where it lands.

In [7]:
sweep = pd.DataFrame([
    dict(band=f"±{b:.0%}", n=e.n, slope=e.slope, intercept=e.intercept, r2=e.r2,
         median_gap=e.median_gap, median_gap_pct=e.median_gap_pct)
    for b in (0.01, 0.02, 0.03, 0.05, 0.10, 0.14, 0.20)
    for e in [mid_vs_print(tape, params, band=b)]
])
print(sweep.to_string(index=False))
print()
print(f"slope {sweep['slope'].min():.4f} - {sweep['slope'].max():.4f}   "
      f"R2 {sweep['r2'].min():.4f} - {sweep['r2'].max():.4f}")
print(f"the band stops binding once n saturates at {sweep['n'].max():,}")

band     n    slope  intercept       r2  median_gap  median_gap_pct
 ±1%  4814 1.001981  -0.007443 0.996327       0.030          0.9036
 ±2%  9188 1.001896  -0.005672 0.994763       0.035          1.2658
 ±3% 12576 0.999664   0.003622 0.994783       0.035          1.5849
 ±5% 16626 0.997860   0.010391 0.996214       0.035          1.8868
±10% 18623 0.998413   0.005617 0.997753       0.045          1.8500
±14% 18712 0.998348   0.006063 0.997880       0.045          1.8448
±20% 18712 0.998348   0.006063 0.997880       0.045          1.8448

slope 0.9979 - 1.0020   R2 0.9948 - 0.9979
the band stops binding once n saturates at 18,712


### 5.4 · The ten bars that actually matter

The fit above is a statement about 18,000 contract-bars. The book only ever filled on **ten**
of them — one call written per week — so the sharpest version of the question is: at those ten
bars, how far was the mid from the print?

Every one of the ten is in the sample, which is not automatic: it needs the chosen strike to
have been both quoted *and* traded in that hour. (A test pins it contract by contract, so a
band or a window that quietly excluded a fill would fail rather than flatter.)

In [8]:
book = run_backtest(tape, params)
sells = book.blotter[(book.blotter["side"] == "SELL")
                     & (book.blotter["instrument"] != params.underlying)]
filled = set(zip(sells["instrument"], sells["time"]))

keys = list(zip(pts["ric"], pd.to_datetime(pts["ts"]).dt.strftime("%Y-%m-%d %H:%M")))
at_fills = pts[[k in filled for k in keys]][
    ["ts", "ric", "strike", "spot", "bid", "ask", "mid", "trdprc_1", "gap", "gap_pct"]
]
print(at_fills.to_string(index=False))

gap = at_fills["gap"].abs()
premium = at_fills["mid"] * 100
print(f"\nfills measured: {len(at_fills)} of {len(sells)}")
print(f"median |print − mid|: ${gap.median():.4f} "
      f"({at_fills['gap_pct'].median():.2f}% of the mid), worst ${gap.max():.3f}")
print(f"per contract that is ${gap.median() * 100:.2f} against a median premium of "
      f"${premium.median():,.2f}")

                       ts                 ric  strike   spot  bid  ask   mid  trdprc_1    gap  gap_pct
2026-07-06 15:00:00-04:00 QQQG102672300.U^G26   723.0 722.27 6.68 6.84 6.760      6.70 -0.060 0.887574
2026-07-13 15:00:00-04:00 QQQG172671200.U^G26   712.0 711.92 8.06 8.24 8.150      8.17  0.020 0.245399
2026-07-20 15:00:00-04:00 QQQG242669600.U^G26   696.0 695.80 8.55 8.64 8.595      8.45 -0.145 1.687027
2026-07-27 15:00:00-04:00 QQQG312668300.U^G26   683.0 682.03 9.09 9.29 9.190      9.25  0.060 0.652884
2026-08-03 15:00:00-04:00 QQQH072670000.U^H26   700.0 699.86 6.73 6.80 6.765      6.83  0.065 0.960828
2026-08-10 15:00:00-04:00 QQQH142672100.U^H26   721.0 720.77 6.11 6.15 6.130      6.15  0.020 0.326264
2026-08-17 15:00:00-04:00 QQQH212673000.U^H26   730.0 729.82 5.03 5.10 5.065      5.10  0.035 0.691017
2026-08-24 15:00:00-04:00 QQQH282670700.U^H26   707.0 706.29 6.10 6.16 6.130      6.10 -0.030 0.489396
2026-08-31 15:00:00-04:00 QQQI042671700.U^I26   717.0 716.87 4.96 5.02 4.

### 5.5 · What this justifies, and what it does not

**Justified.** Filling a one-contract covered call at the bar's midpoint is not optimistic on
this tape. Across the near-the-money sample the mid is an unbiased estimate of the print
(slope ≈ 1, intercept ≈ 0), and at the ten bars the book actually traded on, the median
distance between the two was under four cents — a few dollars per contract against premiums in
the hundreds. Whatever moves the result of this backtest, it is not the fill rule.

**Not justified.** Four limits, all of which belong in the write-up rather than in a reader's
discovery:

1. **This is not a claim about execution.** It measures where a *print* landed relative to a
   *quote*, not whether an order for one contract would have been filled at the mid. A single
   contract is small enough that mid-ish fills are plausible; nothing here proves it.
2. **The pair is not simultaneous**, and the scatter cannot separate "the mid is a good
   estimate" from "the quote moved toward the trade within the hour".
3. **The sample is the contracts that traded** — but the censoring is not where you would
   guess. The unprinted bars are over 95% *in the money*, with a median mid near \$27 and a
   median half-spread more than ten times the printed bars'. Near the money, where the rule
   actually writes, essentially every quoted bar printed. So the gap is a floor on the error
   for deep-ITM contracts and very nearly the whole truth for the ones the strategy touches.
4. **`n` is not `n` independent observations.** The sample is a panel — a few hundred contracts
   observed at hundreds of bars, every contract in a bar sharing one spot and one market
   state — so the R² should not be read as though 16,000 draws stood behind it, and "a slope
   indistinguishable from one" is a claim this sample actually rejects. The defensible
   statement is the *size* of the deviation, not its significance.

**The bound worth quoting is priced, not inferred.** The signed gap has a median of exactly
zero, so the mid is not systematically against the seller. And the worst case can be costed
directly: sell all ten calls at the **bid** instead of the mid and see what the window return
does.

In [9]:
signed = pts["gap"]
print(f"signed gap: mean ${signed.mean():+.4f}, median ${signed.median():+.4f}")
print(f"at the ten fills: mean ${at_fills['gap'].mean():+.4f}, "
      f"median ${at_fills['gap'].median():+.4f}")

cost = ((at_fills["mid"] - at_fills["bid"]) * 100).sum()
worst_nav = book.final_nav - cost
print()
print(f"selling all ten at the bid instead of the mid: ${cost:,.2f}")
print(f"final NAV ${book.final_nav:,.2f} -> ${worst_nav:,.2f}")
print(f"window return {book.total_return:.3%} -> {worst_nav / params.start_cash - 1:.3%}")

signed gap: mean $-0.0078, median $+0.0000
at the ten fills: mean $-0.0065, median $+0.0150

selling all ten at the bid instead of the mid: $49.50
final NAV $76,243.50 -> $76,194.00
window return 1.658% -> 1.592%


In [10]:
print(NON_SIMULTANEITY_CAVEAT)

Within an hourly bar the print is the last trade and the quote is the last bid and ask reported in that bar, so a point is not a simultaneous pair. This is the honest measure of how far a mid sits from a real print at hourly resolution — the resolution the backtest fills at — not a claim that the two were observed at the same instant.
